# 2_models/01 — Light feature-comparison modalities

Notebook fallback for `slurm/launch_feature_comp.sh`, covering the three cheapest modalities.

| Modality | Where it runs |
|---|---|
| `stage`, `treatment`, `metburden` | **here** (1 CPU each) |
| `somatic`, `text`, `prs` | left to the SLURM arrays |

`run_feature_comp_task.py` forces `n_jobs=1` for any modality with fewer than 50 penalized columns,
so these three gain nothing from the cluster's parallelism — they are single-core work queued behind
the heavy fits. `somatic` is nominally in the same `small` class but its design matrix is a wide
gene-by-alteration panel of data-dependent width, so it stays on SLURM.

**Safe to run alongside the arrays.** `run_feature_comp_task.py` skips any scheme/event/modality
whose four output files already exist, so with `OVERWRITE = False` each side steps around the
other's finished work. That check happens once at task start, so a triple begun simultaneously on
both sides is computed twice — deterministic outputs, so this costs CPU rather than correctness. The
`small` array still owns `somatic`, so keep it running rather than `scancel`-ing it.

One subprocess per (event, modality) rather than `--modality all`: `all` means all *six*, which
would pull in exactly the `text` and `prs` work being left to SLURM.

Run this in a compute-backed Jupyter session — **not** on an ERISTwo login node.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import time
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file() and (candidate / "pipelines").is_dir():
            return candidate
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import config  # noqa: E402


def check_inputs(preconditions: list[tuple[str, str]]) -> list[str]:
    """Report presence of each (label, path). Returns the missing labels; never raises."""
    missing = []
    for label, path in preconditions:
        ok = os.path.exists(path)
        if not ok:
            missing.append(label)
        print(f"[{'ok ' if ok else 'MISSING'}] {label:<22} {path}")
    print(f"\n{'All inputs present.' if not missing else str(len(missing)) + ' missing: ' + ', '.join(missing)}")
    return missing


def report_outputs(outputs: list[tuple[str, str]]) -> None:
    """Print size and mtime for each (label, path) that exists."""
    for label, path in outputs:
        if os.path.exists(path):
            mb = os.path.getsize(path) / 1e6
            mtime = time.strftime("%Y-%m-%d %H:%M", time.localtime(os.path.getmtime(path)))
            print(f"[ok     ] {label:<26} {mb:>8.1f} MB   {mtime}")
        else:
            print(f"[missing] {label:<26} {path}")


def run_module(module: str, args: list[str] | None = None, env: dict | None = None,
               capture: bool = False) -> dict:
    """Run `python -m module` from REPO_ROOT. Returns {returncode, wall_s, stdout}."""
    cmd = [sys.executable, "-m", module, *(args or [])]
    started = time.perf_counter()
    run_env = {**os.environ, "PYTHONUNBUFFERED": "1", **(env or {})}
    kwargs = dict(cwd=str(REPO_ROOT), env=run_env)
    if capture:
        kwargs.update(text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    proc = subprocess.run(cmd, **kwargs)
    return {"returncode": proc.returncode, "wall_s": time.perf_counter() - started,
            "stdout": (proc.stdout or "") if capture else ""}


print(f"repo root: {REPO_ROOT}")
print(f"Python:  {sys.executable}")

## Configuration

`MAX_ITER` and `BACKEND` mirror the launcher's `COXNET_MAX_ITER` / `COXNET_BACKEND` defaults; the
grid itself is left at the script's defaults so results stay comparable with the array's.

In [ ]:
import schemes
from anchors import anchor_suffix
from pipelines.training.slurm_array_utils import _feature_file

MODULE = "pipelines.training.run_feature_comp_task"
MODALITIES = ["stage", "treatment", "metburden"]   # subset of MODALITY_CLASS=small

ANCHOR = "treatment"
N_JOBS = 1          # the script forces 1 for <50 penalized cols regardless
MAX_ITER = 1000
BACKEND = "threading"
OVERWRITE = False   # False leaves SLURM's finished work alone
VERBOSE = False     # True replays each subprocess's captured output when it finishes

# Each subprocess is single-core, so speedup tracks worker count up to the core count.
# None = auto: leave a core free, capped at ~4G/task (the SLURM `small` class sizing).
MAX_WORKERS = 8

# Trial-run controls. Both None for the full manifest.
MAX_TASKS = None            # e.g. 4 for a smoke run
SCHEMES_FILTER = None       # e.g. {"death_met"}

MANIFEST = REPO_ROOT / "slurm" / "slurm_manifests" / f"feature_comp_tasks{anchor_suffix(ANCHOR)}.tsv"

print(f"anchor:     {ANCHOR}")
print(f"modalities: {', '.join(MODALITIES)}")
print(f"manifest:   {MANIFEST}")
print(f"overwrite:  {OVERWRITE}   workers: {MAX_WORKERS or 'auto'}")

## Preconditions

The manifest plus the feature files these three modalities read. `complete_somatic_data_df.csv.gz`
and `complete_germline_data_df.csv.gz` are deliberately absent — those are `somatic` and `prs`,
which stay on SLURM. This cell does not raise.

In [ ]:
check_inputs([
    ("manifest",          str(MANIFEST)),
    ("cancer types",      _feature_file("cancer_type_df.csv.gz", ANCHOR)),
    ("cancer stage",      _feature_file("cancer_stage_df.csv.gz", ANCHOR)),
    ("treatment by line", _feature_file("categorical_treatment_data_by_line.csv.gz", ANCHOR)),
    ("met burden",        _feature_file("met_burden_df.csv.gz", ANCHOR)),
])

## Build the task list

The manifest is `scheme<TAB>event`, with an optional third field pinning a row to one modality. A
pinned row is honoured here and dropped when it names a modality left to SLURM.

In [ ]:
SCHEME_ALIASES = {"icd3": "icd3_post", "icd4": "icd4_post", "phecode": "phecode_post"}

tasks: list[tuple[str, str, str]] = []
rows = skipped_pinned = 0

for line_number, raw in enumerate(MANIFEST.read_text().splitlines(), 1):
    if not raw.strip():
        continue
    fields = raw.split("\t")
    if len(fields) not in (2, 3) or not all(fields[:2]):
        raise ValueError(f"{MANIFEST}:{line_number}: expected scheme<TAB>event[<TAB>modality]")
    scheme = SCHEME_ALIASES.get(fields[0], fields[0])
    event = fields[1]
    pinned = fields[2] if len(fields) == 3 and fields[2] else None
    rows += 1

    if pinned is not None:
        if pinned in MODALITIES:
            tasks.append((scheme, event, pinned))
        else:
            skipped_pinned += 1
        continue

    if SCHEMES_FILTER and scheme not in SCHEMES_FILTER:
        continue
    tasks.extend((scheme, event, modality) for modality in MODALITIES)

if SCHEMES_FILTER:
    tasks = [task for task in tasks if task[0] in SCHEMES_FILTER]
if MAX_TASKS is not None:
    tasks = tasks[:MAX_TASKS]

print(f"manifest rows: {rows}")
if skipped_pinned:
    print(f"pinned to a modality left to SLURM: {skipped_pinned}")
print(f"tasks: {len(tasks)}")

## Pre-flight filtering

Three gates, applied **before** the run loop so the queue holds only tasks that will do work. Each
reproduces a check `run_feature_comp_task.py` already makes internally — the point is to make it
before paying for a process launch and a modality's feature load.

1. **Existing results** — the script's own skip test (three grid files plus held-out risk scores).
   This is where the SLURM array's finished work gets subtracted. Read-only: uses
   `schemes.scheme_results_dir`, not `get_output_dir`, which would create directories.
2. **Event exists** — the manifest is a static TSV listing events some schemes do not define.
   `main()` raises before any fitting, so these are cheap but guaranteed failures.
3. **Event prevalence** — `validate_cox_inputs` raises below `MIN_EVENTS_FOR_CV` positives or
   `MIN_NON_EVENTS_FOR_CV` censored. Screening here removes N_MODALITIES processes per
   underpowered event.

**Counting on the right cohort.** Prevalence must be counted on the *common feature cohort* —
`load_feature_modalities_df` restricts every single-modality run to the MRN intersection of the
somatic, PRS, stage and treatment files, then inner-joins `cancer_type_df`. That is substantially
smaller than the raw parquet, so counting before it overestimates and lets underpowered events
through. Reproducing it reads one `DFCI_MRN` column from each of five files, is modality-independent,
and is cached for the notebook.

Still optimistic in one direction: the script drops further rows with NaNs in the modality's own
feature columns, which this gate cannot know without loading them. An event clearing the floor here
can still fail downstream — the gate never drops a task that would have run. Unreadable inputs
**keep** the task and flag it, so they surface as real failures rather than silent skips.

In [ ]:
from functools import lru_cache

import polars as pl

from pipelines.training.slurm_array_utils import MIN_EVENTS_FOR_CV, MIN_NON_EVENTS_FOR_CV
from shared.polars_utils import finite_or_zero


# --- Gate 1: existing results ------------------------------------------------

def missing_outputs(scheme: str, event: str, modality: str) -> list[str]:
    """Which of the four files run_feature_comp_task checks before skipping are absent."""
    comp_dir = os.path.join(schemes.scheme_results_dir(scheme, ANCHOR), "feature_comps", event)
    paths = [
        os.path.join(comp_dir, f"{modality}_test.csv"),
        os.path.join(comp_dir, f"{modality}_val.csv"),
        os.path.join(comp_dir, f"{modality}_ipcw_reference.csv.gz"),
        os.path.join(schemes.feature_held_out_dir(scheme, event, ANCHOR),
                     f"{modality}_risk_scores.csv"),
    ]
    return [os.path.basename(p) for p in paths if not os.path.exists(p)]


def census(task_list: list[tuple[str, str, str]], header: str) -> list[tuple[str, str, str]]:
    done_by_modality = {modality: 0 for modality in MODALITIES}
    remaining, partial = [], []
    for scheme, event, modality in task_list:
        absent = missing_outputs(scheme, event, modality)
        if not absent:
            done_by_modality[modality] = done_by_modality.get(modality, 0) + 1
        else:
            remaining.append((scheme, event, modality))
            # Partial: the script re-runs these, but *reuses* the grid when all three grid
            # files exist -- so one missing only risk scores re-runs cheaply. Worth seeing:
            # a task that produced results earlier and now reads partial means outputs were
            # lost or only partly written.
            if len(absent) < 4:
                partial.append((scheme, event, modality, absent))

    print(f"{header}: {len(task_list) - len(remaining)}/{len(task_list)} complete, "
          f"{len(remaining)} remaining")
    for modality in MODALITIES:
        n_total = sum(1 for task in task_list if task[2] == modality)
        if n_total:
            print(f"  {modality:<10} {done_by_modality[modality]:>5}/{n_total} done")

    if partial:
        print(f"\n  {len(partial)} task(s) partially complete (will re-run):")
        for scheme, event, modality, absent in partial[:15]:
            note = ("grid re-fit" if any(not f.endswith("_risk_scores.csv") for f in absent)
                    else "grid reused, risk scores only")
            print(f"    {scheme}:{event}:{modality:<10} missing {', '.join(absent)}  [{note}]")
        if len(partial) > 15:
            print(f"    ... and {len(partial) - 15} more")
    return remaining


# --- The cohort the script actually fits on ----------------------------------

@lru_cache(maxsize=None)
def analysis_cohort_mrns() -> frozenset:
    """MRNs surviving load_feature_modalities_df's cohort restriction.

    Mirrors _get_common_feature_mrns (somatic & prs & stage & treatment) followed by the
    inner join against cancer_type_df. metburden is excluded from the intersection there by
    design -- it is zero-filled to the full cohort and left-joined.
    """
    def mrns(path: str) -> set:
        return set(pl.read_csv(path, columns=["DFCI_MRN"])["DFCI_MRN"])

    return frozenset(
        mrns(_feature_file("complete_somatic_data_df.csv.gz", ANCHOR))
        & mrns(os.path.join(config.FEATURE_PATH, "complete_germline_data_df.csv.gz"))
        & mrns(_feature_file("cancer_stage_df.csv.gz", ANCHOR))
        & mrns(_feature_file("categorical_treatment_data_by_line.csv.gz", ANCHOR))
        & mrns(_feature_file("cancer_type_df.csv.gz", ANCHOR))
    )


# --- Gates 2 and 3: event exists, and is powered enough ----------------------

@lru_cache(maxsize=None)
def _scheme_parquet(scheme: str) -> str:
    return os.path.join(config.SURV_PATH, schemes.embedding_file(scheme, ANCHOR))


@lru_cache(maxsize=None)
def _scheme_columns(scheme: str) -> frozenset:
    return frozenset(pl.scan_parquet(_scheme_parquet(scheme)).collect_schema().names())


@lru_cache(maxsize=None)
def event_counts(scheme: str, event: str) -> tuple[int, int] | None:
    """(n_events, n_non_events) on the analysis cohort, or None if the event is absent.

    Mirrors the cohort restriction then filter_event_rows: finite positive tt_, finite
    indicator, brain primaries excluded for brainM.
    """
    tt_col = f"tt_{event}"
    available = _scheme_columns(scheme)
    if tt_col not in available or event not in available:
        return None

    wanted = ["DFCI_MRN", event, tt_col]
    needs_brain = event == "brainM" and "CANCER_TYPE_BRAIN" in available
    if needs_brain:
        wanted.append("CANCER_TYPE_BRAIN")

    mask = (
        pl.col(tt_col).cast(pl.Float64, strict=False).is_finite()
        & (pl.col(tt_col) > 0)
        & pl.col(event).cast(pl.Float64, strict=False).is_finite()
    )
    if needs_brain:
        mask = mask & (finite_or_zero("CANCER_TYPE_BRAIN").cast(pl.Boolean) == False)  # noqa: E712

    counts = (
        pl.scan_parquet(_scheme_parquet(scheme)).select(wanted)
        .filter(pl.col("DFCI_MRN").is_in(analysis_cohort_mrns()))
        .filter(mask)
        .select(pl.len().alias("n_rows"),
                pl.col(event).cast(pl.Float64, strict=False).sum().alias("n_events"))
        .collect()
    )
    n_rows = int(counts["n_rows"][0])
    n_events = int(counts["n_events"][0] or 0)
    return n_events, n_rows - n_events


def prevalence_filter(task_list):
    """Split tasks into (runnable, unrunnable) on event existence and CV prevalence."""
    verdicts: dict[tuple[str, str], tuple[str, str]] = {}
    keep, dropped = [], []

    for scheme, event, modality in task_list:
        key = (scheme, event)
        if key not in verdicts:
            try:
                counts = event_counts(scheme, event)
            except Exception as exc:  # unreadable input -> keep the task, surface the error
                verdicts[key] = ("unchecked", f"{type(exc).__name__}: {exc}")
            else:
                if counts is None:
                    # main() raises "Event not found" before fitting -- a definitive drop.
                    verdicts[key] = ("absent", "not defined for this scheme")
                else:
                    n_events, n_non_events = counts
                    if n_events < MIN_EVENTS_FOR_CV:
                        verdicts[key] = ("underpowered", f"{n_events} events < {MIN_EVENTS_FOR_CV}")
                    elif n_non_events < MIN_NON_EVENTS_FOR_CV:
                        verdicts[key] = ("underpowered",
                                         f"{n_non_events} censored < {MIN_NON_EVENTS_FOR_CV}")
                    else:
                        verdicts[key] = ("ok", f"{n_events} events / {n_non_events} censored")
        status = verdicts[key][0]
        (dropped if status in ("absent", "underpowered") else keep).append((scheme, event, modality))

    unchecked = {k: r for k, (st, r) in verdicts.items() if st == "unchecked"}
    if unchecked:
        print(f"{len(unchecked)} event(s) could not be checked — kept in the queue:")
        for (scheme, event), reason in sorted(unchecked.items()):
            print(f"  {scheme}:{event} — {reason}")

    for status, title in (("absent", "not defined for their scheme"),
                          ("underpowered", "below the CV minimums")):
        events = sorted(k for k, (st, _) in verdicts.items() if st == status)
        if not events:
            continue
        n_tasks = sum(1 for s, e, _ in dropped if verdicts[(s, e)][0] == status)
        print(f"\n{len(events)} event(s) {title} — dropping {n_tasks} task(s):")
        for scheme, event in events[:15]:
            print(f"  {scheme}:{event:<12} {verdicts[(scheme, event)][1]}")
        if len(events) > 15:
            print(f"  ... and {len(events) - 15} more")

    print(f"\nPrevalence: {len(keep)}/{len(task_list)} tasks runnable, {len(dropped)} dropped")
    return keep, dropped


not_done = census(tasks, "Existing results")
print(f"\nAnalysis cohort: {len(analysis_cohort_mrns()):,} patients")
pending, skipped_prevalence = prevalence_filter(not_done)

print(f"\nQueue: {len(pending)} task(s) ({len(tasks)} manifest "
      f"- {len(tasks) - len(not_done)} done - {len(skipped_prevalence)} unrunnable)")

## Run

One subprocess per **pending** task, run concurrently across `MAX_WORKERS` processes. Concurrency is
safe because each task is genuinely single-core: BLAS is pinned to one thread (matching
`array_feature_comp.sh`) and the script forces `n_jobs=1` for all three modalities.
`POLARS_MAX_THREADS`/`RAYON_NUM_THREADS` are pinned too — *not* in the SLURM script, where one task
owns the node, but essential here: Polars sizes its Rayon pool to the core count at import, which
with several tasks in flight is hundreds of threads and trips "can't start new thread" before any
fitting begins.

`MAX_WORKERS = None` auto-sizes to one less than the CPU count, capped at ~4G/worker. On a shared
node, size it to what you actually reserved, not what `os.cpu_count()` reports.

`VERBOSE = False` collapses the run to a progress bar. Each subprocess's stdout is still **captured**
— it is the only record of why a task failed, and the summary cell reads it — just not echoed, since
under concurrency live streaming would interleave unreadably.

In [ ]:
import concurrent.futures

from tqdm.auto import tqdm


def resolve_workers(n_tasks: int) -> int:
    """Auto-size the pool: leave a core free, and budget ~4G/task (the SLURM small class)."""
    if MAX_WORKERS is not None:
        return max(1, min(int(MAX_WORKERS), n_tasks))
    workers = max(1, (os.cpu_count() or 1) - 1)
    try:
        # Linux-only; on a cgroup-limited node this reflects the real allowance better
        # than total system memory. Skipped silently where unavailable.
        available_gb = (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_AVPHYS_PAGES")) / 1024 ** 3
        workers = max(1, min(workers, int(available_gb // 4)))
    except (ValueError, OSError, AttributeError):
        pass
    return max(1, min(workers, n_tasks))


def run_task(scheme: str, event: str, modality: str) -> dict:
    args = ["--scheme", scheme, "--event", event, "--modality", modality,
            "--anchor", ANCHOR, "--n-jobs", str(N_JOBS),
            "--max-iter", str(MAX_ITER), "--backend", BACKEND]
    if OVERWRITE:
        args.append("--overwrite")
    single_thread = {var: "1" for var in (
        "OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS",
        "POLARS_MAX_THREADS", "RAYON_NUM_THREADS")}
    outcome = run_module(MODULE, args, env=single_thread, capture=True)
    outcome.update(scheme=scheme, event=event, modality=modality)
    return outcome


# OVERWRITE re-runs finished work by design, so it bypasses the existing-results gate.
# The prevalence gate still applies -- --overwrite does not make an underpowered event fittable.
_underpowered = set(skipped_prevalence)
queue_tasks = pending if not OVERWRITE else [t for t in tasks if t not in _underpowered]
n_workers = resolve_workers(len(queue_tasks))
print(f"Running {len(queue_tasks)} task(s) across {n_workers} worker(s)"
      + (" (OVERWRITE=True)" if OVERWRITE else ""))

run_started = time.perf_counter()
results, n_failed = [], 0

if queue_tasks:
    # Threads only wait on subprocess.run -- the work is in separate processes, so the GIL
    # is irrelevant and a thread pool avoids pickling overhead.
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        in_flight = {pool.submit(run_task, *task): task for task in queue_tasks}
        with tqdm(total=len(queue_tasks), desc="feature comps", unit="task") as bar:
            try:
                for future in concurrent.futures.as_completed(in_flight):
                    result = future.result()
                    results.append(result)
                    label = f"{result['scheme']}:{result['event']}:{result['modality']}"
                    if result["returncode"] != 0:
                        n_failed += 1
                        bar.write(f"[fail] {label} (exit {result['returncode']})")
                    if VERBOSE:
                        bar.write(f"\n{'=' * 72}\n{label} (exit {result['returncode']}, "
                                  f"{result['wall_s']:.1f}s)\n{'=' * 72}")
                        bar.write(result["stdout"].rstrip())
                    bar.update(1)
                    bar.set_postfix_str(f"{n_failed} failed" if n_failed else "", refresh=True)
            except KeyboardInterrupt:
                # Without this, pool shutdown would block on every queued task.
                for future in in_flight:
                    future.cancel()
                print("\nInterrupted -- cancelled queued tasks, waiting for in-flight ones.")
                raise

run_elapsed = time.perf_counter() - run_started
print(f"\nRan {len(results)} task(s) in {run_elapsed / 60:.1f} min across {n_workers} worker(s)")

## Summary

With `VERBOSE = False` this is where a failure is diagnosed — the run loop captured every
subprocess's output but printed none of it, so the tail of each failing task's stdout is reproduced
here. The full text stays in `results[i]["stdout"]`.

With both gates applied, a non-zero exit is more likely to be a real defect: already-done work and
underpowered events were removed before the loop started. What remains is post-imputation row loss
(the prevalence gate counts before modality NaN drops) and genuine failures;
`results/skipped_events/*.jsonl` records the reason either way.

The final census runs over the full manifest, so "complete" includes whatever the SLURM arrays
finished while this notebook was running.

In [ ]:
FAIL_TAIL_LINES = 15

failed = [r for r in results if r["returncode"] != 0]
print(f"{len(results) - len(failed)} succeeded, {len(failed)} failed")

for result in failed:
    label = f"{result['scheme']}:{result['event']}:{result['modality']}"
    lines = result["stdout"].splitlines()
    tail = lines[-FAIL_TAIL_LINES:]
    print(f"\n{'-' * 72}\n{label} (exit {result['returncode']}) — "
          f"last {len(tail)} of {len(lines)} output line(s)\n{'-' * 72}")
    print("\n".join(tail) if tail else "(no output captured)")

if results:
    # Per-task wall time: with n_workers in flight these sum to more than the elapsed clock.
    total_task_min = sum(r["wall_s"] for r in results) / 60
    print(f"\nSlowest tasks (total task time {total_task_min:.1f} min across "
          f"{run_elapsed / 60:.1f} min wall):")
    for result in sorted(results, key=lambda r: r["wall_s"], reverse=True)[:5]:
        print(f"  {result['wall_s'] / 60:>6.1f} min  "
              f"{result['scheme']}:{result['event']}:{result['modality']}")

print()
census(tasks, "After run")